In [16]:
#TODO make sure this renders in the github repo

# Configurations For The Llama 3 Models

![chart](./showcase_images/model_sizes_chart.png)

- **Layers:** Noted as **num_layers**. How many Decoder layers the model has.
- **Model Dimensions:** Also noted as **d_model** or $d_{model}$. It represents the expected features for each token in the input and output sequences.
  - Example: when the Token Embeddings layer happen, each token is converted into a dense vector of size $d_{model}$.
- **FFN Dimensions:** Noted as **d_ff**. Is the hidden size of the Feed Forward SwiGLU sub-layer. We use this to expand the linear layers for SwiGLU, which gives the model more parameters to learn non-linear functions. 
- **Attention Heads:** This the the total number of Query heads, note K & V use the Key/Value Heads hyperparameter.
  - If d_model = $4{,}096$ and attn_heads = $32$, each individual head processes a vector of size $4{,}096/32 = 128$ 
- **Key/Value Heads:** Noted as **num_kv_heads**. The number of key and value heads in the attention mechanism. The reason the number of Key and Value heads is smaller than the number of Query heads, is because this allows faster generation, larger batch sizes, and reduces the memory footprint.
- **Peak Learning Rate:** At the end of the warmup, the learning rate hits the peak it can be, then it starts to decay. This keeps training stable, and prevents exploding gradients early on when the weights are random.

- **Vocabulary Size:** The model's dictionary. It represents the words/tokens model can understand. It is static.
    - **Context Window length (context_len/sequence_len):** The model's short-term memory. Represents how much words/tokens the model can remember at one time. So, as you chat with the model, it loses track of earlier context that no longer fits its context window. In the data, it is the number of tokens in a single document/row in a list of documents.
      - **Micro Batch Size:** The number of sequences stacked together in a single batch.
      - Example: `micro_batch_size` $=4$ and `context_len` $=4{,}096$, a single batch will contain $4*4{,}096 = 16{,}384 \, \text{tokens}$.

- **Positional Embeddings** The frequency that the Q and K tokens are rotated by, to give them positional information.

The [Llama-3.1 8B tokenizer](https://huggingface.co/meta-llama/Llama-3.1-8B/blob/main/tokenizer_config.json) consists of $128{,}255$ tokens.
- **Standard tokens:** IDs $0$ through $127{,}999$ ($128{,}000$ total) are the standard tokens.
- **Special tokens:** IDs $128{,}000$ through $128{,}255$ ($256$ total) are the special tokens.
- **From Llama 3 paper:** "We use a vocabulary with 128K tokens. Our token vocabulary combines 100K tokens from the tiktoken tokenizer with 28K additional tokens to better support non-English languages. Compared to the Llama 2 tokenizer, our new tokenizer improves compression rates on a sample of English data from 3.17 to 3.94 characters per token. This enables the model to “read” more text for the same amount of training compute. We also found that adding 28K tokens from select non-English languages improved both compression ratios and downstream performance, with no impact on English tokenization."

**Token Budget:**
- In LLM, data is not measured in gigabytes, sentence pairs, or epochs. Compute and training length are measured in tokens. The token budget dictates the maximum number of tokens you want the model to process before training terminates:

$$\text{Token Budget} = \text{Global Batch Size(in tokens)} * \text{Total Training Steps}$$

Example: $\text{Token Budget} = 1.5 \text{Billion}$ tokens, and your hardware can handle a $\text{Global Batch Size}$ of $500{,}000$ tokens per step, then: $1.5 \text{Billion} / 500{,}000 = 3{,}000$ total training steps.

- **Total Training Steps**: The number of times the model's weights are updated.
- **Global Batch Size**: The total number of tokens the optimizer needs to look at together to calculate an accurate gradient. In code, it dictates when `optimizer.step()` is called.
  - Without **Gradient Accumulation**: $\text{Global Batch Size} = \text{Micro Batch Size} \times \text{Context Length}$
  - With **Gradient Accumulation**: $\text{Global Batch Size} = \text{Micro Batch Size} \times \text{Context Length} \times \text{Accumulation Steps}$
    - **Gradient Accumulation Steps**: The number of sequential forward and backward passes over a micro-batch the model executes before aggregating the gradients and updating the weights via `optimizer.step()`. This allows you to simulate a massive Global Batch Size that would otherwise not fit in your hardware's VRAM.
  - For this implementation, I will use gradient accumulation. And you only ever have to worry about the `micro_batch_size`, the config will handle setting the `total_training_steps` and `global_batch_size` based on the `micro_batch_size`, `context_len`, and `gradient_accumulation_steps`, and `token_budget`.

- **Micro Batch Size:** The number of sequences your hardware can hold in its VRAM (GPU Memory) during a single forward and backward pass.


## Configurations:

**Note:** All the models other than my scaled down version is be to be used to loaded Meta's pre-trained Llama 3 models.

In [ ]:
import os
import EasyJupyter
from pathlib import Path
import torch
from typing import TypeVar, TYPE_CHECKING
import json

T = TypeVar("T", bound="BaseConfig")
if TYPE_CHECKING:
    from model.tokenizer import BPETokenizer

class BaseConfig:
    """
    The is the base config class that all models inherit from.
    It should never be instantiated, only its child classes should be!

    The parameters are defined in the markdown cells of llama_configs.ipynb
    """

    @staticmethod
    def _find_root() -> Path:
        """Find the root directory of the project, by climbing up the directory tree"""
        curr = Path.cwd().resolve()
        for parent in [curr] + list(curr.parents):
            if parent.name == "How_to_build_an_LLM":
                return parent
        return curr


    model_summary_input = "Not yet set"  # Asks the user when ran.

    # All Llama models use the same below parameters, except for my scale down model.
    num_kv_heads = 8 
    activation_function = "SwiGLU"
    pos_embeddings_freq = 500_000  # RoPE frequency

    # Force child classes to implement the below attributes
    _required_attributes: dict[str, type] = {
        "model_name": str,
        "context_len": int,
        "num_layers": int,  # How many decoder layers the model has.
        "d_model": int,
        "d_ff": int,  # How much to expand the SwiGLU linear to.
        "attn_heads": int,  # The number of independent "Query" viewpoints the attention mechanism splits the data into.
        "rms_norm_eps": float,  # RMSNorm epsilon #TODO set to 1e-5 for most models, maybe higher for my scale down model.
        "peak_lr": float,  # The peak learning rate.
        "micro_batch_size": int,
        "token_budget": int,
        "gradient_accumulation_steps": int,
        "vocab_size": int,
        "warmup_steps": int,
    }

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        for attr, expected_type in cls._required_attributes.items():
            value = getattr(cls, attr, None)
            if value in ("NOT_NEEDED", "# TODO", "#TODO", 000000000):
                continue
            elif value is None:
                raise ValueError(f"Missing attribute {attr} in {cls.__name__}")
            elif not isinstance(value, expected_type):
                raise TypeError(
                    f"Attribute {attr} in {cls.__name__} must be of type {expected_type}, but got {type(value)}"
                )
        pass



    # ================== Training ==================
    ### 🚨 NOTE Continue Training from a checkpoint:
    #       Models I train are saved to ./model/checkpoints/<chpt_dir_name>
    #       That directory will contain the model checkpoint (.pt), its tokenizer (.json), and other files associated with that specific model.
    continue_training = False  # Ask  user on run.
    training_stage = "Not set"  # This is set by the pre-training and post-training scripts. #TODO and make sure to check this when loading the model, it should not be set to --post-training if the config has training stage to pre-training.



    # ================== Dataset ==================
    special_tokens = {
        "pad_token": {
            "token": "<|pad_token|>",
            "ID": 0,
        },
        "doc_begin_token": {  # The token for the beginning of the text.
            "token": "<|begin_of_doc|>",
            "ID": 1,
        },
        "doc_end_token": {  # The token for the end of document, it is injected between documents.
            "token": "<|end_of_doc|>",
            "ID": 2,
        },
        "unk_token": {  # The unknown token.
            "token": "<|unk|>",
            "ID": 3,
        },
    }



    # ================== System ==================
    num_workers = 0  # 🚨 NOTE: For NVIDIA GPUs the Golden Rule is: num_worker = 4 * num_GPU | On Mac Silicone even though I have a 32 core GPU, it is still only one GPU, best to set num_workers = 0.
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")


    PROJECT_ROOT = _find_root()
    DATA_DIR = PROJECT_ROOT / "data"
    MODEL_DIR = PROJECT_ROOT / "model"
    CHPT_DIR = MODEL_DIR / "checkpoints"


    def __init__(self):
        self._setup_folders()


    def _setup_folders(self):
        print("\nProject Root:", self.PROJECT_ROOT)
        folders_to_make = [
            self.DATA_DIR,
            self.MODEL_DIR / "saved_models",
            self.MODEL_DIR / "saved_models" / "meta-models",  # Store Meta-models here
            self.MODEL_DIR / "checkpoints"
        ]
        for folder in folders_to_make:
            folder.mkdir(parents=True, exist_ok=True)

    @property
    def global_batch_size_tokens(self) -> int:
        """
        Get the Global Batch Size which is the total number of tokens processed per optimizer step.
        """
        return (
            self.micro_batch_size * self.context_len * self.gradient_accumulation_steps
        )

    @property
    def total_training_steps(self) -> int:
        """
        Calculates the total number of optimizer steps required to spend the token budget.
        """
        return self.token_budget // self.global_batch_size_tokens

    @classmethod
    def load_config_from_json(cls, chpt_dir_name: str):
        """
        Load the model configuration from a json file
        The loading path e.g., ./model/checkpoints/<model_name>/config.json

        Args:
            cls: An uninitialized config object.
            chpt_dir_name: The name of the directory that contains all the model files. e.g., ./model/checkpoints/<model_name>.
        """
        config_path = cls.CHPT_DIR / chpt_dir_name / "config.json"

        with open(config_path, "r") as f:
            saved_data = json.load(f)

        new_cfg = cls()

        # Update from the config file to the config object
        for key, value in saved_data.items():
            # if isinstance(value, str):
            #     if value in ("cpu", "cuda", "mps"):
            #         # Reconstruct the
            #         value = torch.device(value)
            setattr(new_cfg, key, value)

        print(f"Loaded Config from -> {config_path}")
        return new_cfg

    def save_config_to_json(self):
        """
        Save the model configuration into a json file, so that it can be viewed or loaded later.
        The save path e.g., ./model/checkpoints/<model_name>/config.json
        """
        config_dict = {}
        for attr_name in dir(self):
            if isinstance(getattr(type(self), attr_name, None), property):
                continue

            if attr_name.startswith("_") or attr_name == "folders_to_make":
                # Skip callable methods like _vars, etc..
                # Skip non-serializable attributes like the folders_to_make
                continue

            # Skip the path like MODEL_DIR
            if attr_name.endswith("_DIR") or attr_name.endswith("_ROOT"):
                continue

            value = getattr(self, attr_name)

            if not callable(value):
                # Convert non-JSON serializable types to strings
                if isinstance(value, (Path, torch.device)):
                    config_dict[attr_name] = str(value)
                else:
                    config_dict[attr_name] = value

        config_save_path = self.CHPT_DIR / self.model_name / "config.json"
        config_save_path.parent.mkdir(parents=True, exist_ok=True)

        with open(config_save_path, "w") as f:
            json.dump(config_dict, f, indent=4)

        print(f"Saved Config to -> {config_save_path}")

    def __repr__(self):
        return f"""\n\n
        {'='*20} Config {'='*20}

        [Model Architecture]
          - model_name = {self.model_name}
          - model_summary_input = {self.model_summary_input}
          - d_model = {self.d_model}
          - num_kv_heads = {self.num_kv_heads}
          - context_len = {self.context_len}
          - d_ff = {self.d_ff}
          - num_layers = {self.num_layers}
          - attn_heads = {self.attn_heads}
          - rms_norm_eps = {self.rms_norm_eps}
          - pos_embeddings_freq = {self.pos_embeddings_freq}
        \n
        [Dataset & Tokenizer]
          - vocab_size = {self.vocab_size}
          - token_budget = {self.token_budget}
          - global_batch_size = {self.global_batch_size_tokens}
          - micro_batch_size = {self.micro_batch_size}
        \n
        [Training]
          - total_training_steps = {self.total_training_steps}
          - training_stage = {self.training_stage}
          - num_workers = {self.num_workers}
          - warmup_steps = {self.warmup_steps}
          - activation_function = {self.activation_function}
          - continue_training = {self.continue_training}
          - gradient_accumulation_steps = {self.gradient_accumulation_steps}
          - peak_lr = {self.peak_lr}
        \n\n
        """

**Llama 3 8B Configuration:**

**Llama 3 paper:** "**Language model pre-training.** We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction. In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”. To do this effectively, pre-training is performed at massive scale: we pre-train **a model with 405B parameters** on **15.6T tokens using a context window of 8K tokens**. This standard pre-training stage is followed by a continued pre-training stage that increases the supported **context window to 128K tokens**."

I sourced the configurations for this model from [unsloth/llama-3-8b](https://huggingface.co/unsloth/llama-3-8b/blob/main/config.json) and [meta-llama/Llama-3.1-8B](https://huggingface.co/meta-llama/Llama-3.1-8B/tree/main)

In [ ]:
class Llama3_8B(BaseConfig):
    """
    Llama 3.1 8B Configuration. This model is not multi-modal, it only takes in text!
    """
    # TODO Map the weights from Meta's file containing state dictionary, e.g., model.layers.0.self_attn.q_proj.weight to my instantiated class
    model_name = "Llama_3_8B"
    num_layers = 32
    d_model = 4_096
    d_ff = 14_336
    attn_heads = 32
    num_kv_heads = 8

    peak_lr = 3e-5
    rms_norm_eps= 1e-5
    gradient_accumulation_steps = "NOT_NEEDED" # This is only needed for training, I am loading the pre-trained models
    micro_batch_size =  "NOT_NEEDED"
    context_len = 8192
    token_budget = 15_600_000_000_000
    vocab_size = 128_256
    warmup_steps= 8000

    def __init__(self):
        super().__init__()


In [ ]:
class Llama3_scaled_down(BaseConfig):
    """
    Scaled down version of the Llama 3 architecture that is trainable on my Mac.
    """
    model_name = "Scaled_down_Llama_3"

    num_layers = 8
    d_model = 768
    d_ff = 2048
    num_kv_heads = 4
    attn_heads = 12
    rms_norm_eps= 1e-5
    vocab_size = 32_000
    context_len = 1024

    micro_batch_size = 8
    gradient_accumulation_steps = 16
    peak_lr = 5e-4 # Higher peak learning rate for smaller models
    token_budget = 250_000_000
    warmup_steps = 500
    pos_embeddings_freq = 10_000

    def __init__(self):
        super().__init__()

In [20]:
# @i-c
l = Llama3_scaled_down()


Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM


In [21]:
# TODO Add the config for the larger models, remember to add it to the argparse in main.py